# Practice Day 12 · Conditions, missing values, text and dates

Row-by-row skills: `when`, conditions, nulls, `distinct`, strings and regular expressions, dates, and unions. In Zillacode's list these are the *Conditional Logic*, *Case When Statements*, *Null Handling*, *String Manipulation*, *Regular Expressions*, *Datetime Operations*, *Union Statements* and *Distinct Function* topics.

14 problems, about 2 hours. They prepare you for the Zillacode problems of Days 15 to 26.

## How to work
* Run the **Setup** cell first. `F` is `pyspark.sql.functions` and `Window` is `pyspark.sql.Window`, the same imports Zillacode's editor gives you (it calls the window class `W`).
* Every problem is in Zillacode's format: complete the function `etl(...)` in the cell under it, so that it **returns** a DataFrame, then run the cell. The last line, `etl(...).show()`, prints your result.
* Your result must look exactly like the **expected output**: same columns, same rows, same order. Where an expected output is left-aligned, it was printed with `.show(truncate=False)`, as the cell already does.
* Stuck for 15 minutes? Open the hint under the problem. The answers are collapsed at the very end of the notebook: open one only after you have tried, then write the solution again yourself without looking.

**Setup (once):** Java 17 or 21 and Python 3.10 or newer. In a new folder: `python3.12 -m venv .venv`, `source .venv/bin/activate`, `pip install pyspark==4.2.0 "pandas>=2.2,<3" "pyarrow>=18" jupyterlab`, then `jupyter lab` and open this notebook.

## The data
The same small online shop as in the theory days. The setup cell creates every table, and you can read every row in it.

* **customers**: *customer_id*, *name*, *email*, *city*, *signup_date*
* **products**: *product_id*, *product_name*, *category*, *price* (dollars for one item)
* **orders**: *order_id*, *customer_id*, *product_id*, *quantity*, *order_date*, *status* (`delivered`, `shipped` or `cancelled`)

Good to know: order 1011 is a guest checkout (no *customer_id*); customer 6 has no email and no city; customer 7 has never ordered; product 107 has never been sold; order 1016 is for product 108, which is no longer in *products*.

This notebook also has:
* **reviews**: *review_id*, *product_id*, *customer_id*, *review_date* (text like `15/01/2024`), *rating*, *comment* (messy text), *tags* (text like `fast delivery;good price`)
* **customers_new**: the customers table one week later: some rows changed, one customer is gone, one is new
* **signups_raw**: sign-ups exactly as a web form delivered them: *signup_id*, *full_name*, *email*, *phone*, *signup_date* (text, three formats), *amount* (text), *prefs* (JSON text)

## Setup
Run this cell once. It starts Spark and creates the tables.

In [ ]:
# Setup: starts Spark and creates the shop tables. Run it once.
import logging, os, shutil, sys, time
from datetime import date
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this notebook's Python
from pyspark.sql import SparkSession, Window, functions as F

spark = (SparkSession.builder
         .master("local[4]")   # 4 tasks at a time on your machine
         .appName("practice12")
         .config("spark.sql.shuffle.partitions", "4")          # small data: 4 shuffle partitions are enough
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.log.level", "FATAL")                  # hide Spark's log messages
         # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this
         .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false")
         .getOrCreate())
for name in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
    logging.getLogger(name).setLevel(logging.CRITICAL)        # errors still show, without an extra JSON log

customers_rows = [
    # customer_id, name, email, city, signup_date
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Berlin",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     None,               None,       date(2024, 3, 10)),
    (7, "Grace Lee",        "grace@web.de",     "Berlin",   date(2024, 3, 15)),
]
products_rows = [
    # product_id, product_name, category, price
    (101, "Laptop",       "Electronics", 1200.0),
    (102, "Headphones",   "Electronics",  150.0),
    (103, "Coffee Maker", "Kitchen",       85.0),
    (104, "Blender",      "Kitchen",       60.0),
    (105, "Novel",        "Books",         15.0),
    (106, "Cookbook",     "Books",         25.0),
    (107, "Desk Lamp",    "Home",          40.0),
]
orders_rows = [
    # order_id, customer_id, product_id, quantity, order_date, status
    (1001, 1,    101, 1, date(2024, 1, 10), "delivered"),
    (1002, 1,    105, 3, date(2024, 1, 10), "delivered"),
    (1003, 2,    102, 2, date(2024, 1, 25), "delivered"),
    (1004, 3,    103, 1, date(2024, 2, 3),  "cancelled"),
    (1005, 2,    106, 1, date(2024, 2, 5),  "delivered"),
    (1006, 4,    104, 2, date(2024, 2, 15), "shipped"),
    (1007, 1,    102, 1, date(2024, 2, 20), "delivered"),
    (1008, 5,    101, 1, date(2024, 3, 2),  "shipped"),
    (1009, 3,    105, 4, date(2024, 3, 5),  "delivered"),
    (1010, 4,    106, 2, date(2024, 3, 8),  "cancelled"),
    (1011, None, 104, 1, date(2024, 3, 9),  "delivered"),
    (1012, 6,    103, 2, date(2024, 3, 12), "shipped"),
    (1013, 2,    101, 1, date(2024, 3, 18), "delivered"),
    (1014, 5,    105, 5, date(2024, 3, 20), "delivered"),
    (1015, 1,    103, 1, date(2024, 3, 25), "shipped"),
    (1016, 3,    108, 1, date(2024, 3, 28), "delivered"),
]

# Write the tables as Parquet files and read them back, the way real data arrives.
# data/shop belongs to this lab: it is rebuilt every time you run this cell.
shutil.rmtree("data/shop", ignore_errors=True)
for table, rows, schema in [
    ("customers", customers_rows, "customer_id INT, name STRING, email STRING, city STRING, signup_date DATE"),
    ("products",  products_rows,  "product_id INT, product_name STRING, category STRING, price DOUBLE"),
    ("orders",    orders_rows,    "order_id INT, customer_id INT, product_id INT, quantity INT, order_date DATE, status STRING"),
]:
    spark.createDataFrame(rows, schema).coalesce(1).write.parquet(f"data/shop/{table}")

customers = spark.read.parquet("data/shop/customers")
products = spark.read.parquet("data/shop/products")
orders = spark.read.parquet("data/shop/orders")

# product reviews
reviews = spark.createDataFrame([
    # review_id, product_id, customer_id, review_date (text), rating, comment, tags
    (1, 101, 1, "15/01/2024", 5, "  Great laptop, very FAST!  ", "fast delivery;good price"),
    (2, 105, 1, "20/01/2024", 4, "Nice story.",                  "good price"),
    (3, 102, 2, "02/02/2024", 3, "Sound is ok, delivery was slow", "slow delivery"),
    (4, 103, 3, "10/02/2024", 1, "Broke after one week!!",       "broken;slow delivery"),
    (5, 101, 5, "09/03/2024", 5, "Fast and powerful",            "fast delivery"),
    (6, 105, 3, "12/03/2024", 2, "Too long, boring",             None),
    (7, 106, 2, "25/03/2024", 4, "Useful recipes",               "good price;gift"),
    (8, 103, 6, "30/03/2024", 4, "Good coffee, slow start",      "gift"),
], "review_id INT, product_id INT, customer_id INT, review_date STRING, rating INT, comment STRING, tags STRING")

# the customers table one week later
customers_new = spark.createDataFrame([
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Munich",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     "farid@mail.az",    None,       date(2024, 3, 10)),
    (8, "Hana Sato",        "hana@mail.jp",     "Osaka",    date(2024, 4, 2)),
], customers.schema)

# raw sign-ups, exactly as a web form delivered them
signups_raw = spark.createDataFrame([
    # signup_id, full_name, email, phone, signup_date (text), amount (text), prefs (JSON text)
    (1, "  aziz   KARIMOV ", "Aziz@Mail.UZ ",      "+998 90 123-45-67", "2024-01-05", "12.50", '{"newsletter": true, "lang": "uz"}'),
    (2, "bella schmidt",     "bella@web.de",       "0049 151 2345678",  "20/01/2024", "abc",   '{"newsletter": false, "lang": "de"}'),
    (3, "Carlos  Ruiz",      "",                   "(34) 612 345 678",  "2024/02/02", "7",     '{"lang": "es"}'),
    (4, "DILNOZA rahimova",  "dilnoza@mail.uz",    "90-765-43-21",      "14/02/2024", "",      None),
    (5, "Emma Wilson",       "emma[at]post.co.uk", "07700 900123",      "2024-03-01", "3.75",  '{"newsletter": true}'),
    (6, "aziz karimov",      "aziz@mail.uz",       "+998901234567",     "2024-03-02", "1.00",  '{"newsletter": false, "lang": "uz"}'),
], "signup_id INT, full_name STRING, email STRING, phone STRING, signup_date STRING, amount STRING, prefs STRING")

print("Spark", spark.version, "| Spark UI:", spark.sparkContext.uiWebUrl)

Look at the tables:

In [ ]:
customers.show()
products.show()
orders.show()
reviews.show(truncate=False)
customers_new.show()
signups_raw.show(truncate=False)

## Problems

**Q1 · `etl(products)`** Give every product a `price_level`: `cheap` if the price is under 50, `mid` from 50 to 200, `premium` above 200. Show product_name, price and price_level.

Expected output:

```
+------------+------+-----------+
|product_name| price|price_level|
+------------+------+-----------+
|      Laptop|1200.0|    premium|
|  Headphones| 150.0|        mid|
|Coffee Maker|  85.0|        mid|
|     Blender|  60.0|        mid|
|       Novel|  15.0|      cheap|
|    Cookbook|  25.0|      cheap|
|   Desk Lamp|  40.0|      cheap|
+------------+------+-----------+
```

<details><summary>Hint</summary>

`F.when(condition, value).when(condition, value).otherwise(value)`: the first true condition wins.

</details>

In [ ]:
def etl(products):
    # Write code here: return a DataFrame
    return products

etl(products).show()

**Q2 · `etl(orders)`** Show the orders that are `shipped` or `cancelled` and have a quantity of 2 or more: order_id, status and quantity.

Expected output:

```
+--------+---------+--------+
|order_id|   status|quantity|
+--------+---------+--------+
|    1006|  shipped|       2|
|    1010|cancelled|       2|
|    1012|  shipped|       2|
+--------+---------+--------+
```

<details><summary>Hint</summary>

`Column.isin(...)`. Combine conditions with `&` (and) or `|` (or), and put each comparison in parentheses.

</details>

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q3 · `etl(customers)`** Show customer_id, name and city of every customer, with `Unknown` where the city is missing. Call the name column `customer_name`.

Expected output:

```
+-----------+----------------+--------+
|customer_id|   customer_name|    city|
+-----------+----------------+--------+
|          1|    Aziz Karimov|Tashkent|
|          2|   Bella Schmidt|  Berlin|
|          3|     Carlos Ruiz|  Madrid|
|          4|Dilnoza Rahimova|Tashkent|
|          5|     Emma Wilson|  London|
|          6|    Farid Aliyev| Unknown|
|          7|       Grace Lee|  Berlin|
+-----------+----------------+--------+
```

<details><summary>Hint</summary>

`withColumnRenamed(old, new)`. `F.coalesce(a, b)` returns the first value that is not null; or use `na.fill`.

</details>

In [ ]:
def etl(customers):
    # Write code here: return a DataFrame
    return customers

etl(customers).show()

**Q4 · `etl(customers)`** Which cities do the customers live in? One row per city, no empty values, from A to Z.

Expected output:

```
+--------+
|    city|
+--------+
|  Berlin|
|  London|
|  Madrid|
|Tashkent|
+--------+
```

<details><summary>Hint</summary>

`dropna()` removes rows with nulls, `distinct()` removes repeated rows.

</details>

In [ ]:
def etl(customers):
    # Write code here: return a DataFrame
    return customers

etl(customers).show()

**Q5 · `etl(customers)`** Split `name` into `first_name` and `last_name`. Show customer_id, first_name and last_name.

Expected output:

```
+-----------+----------+---------+
|customer_id|first_name|last_name|
+-----------+----------+---------+
|          1|      Aziz|  Karimov|
|          2|     Bella|  Schmidt|
|          3|    Carlos|     Ruiz|
|          4|   Dilnoza| Rahimova|
|          5|      Emma|   Wilson|
|          6|     Farid|   Aliyev|
|          7|     Grace|      Lee|
+-----------+----------+---------+
```

<details><summary>Hint</summary>

`F.split(column, " ")` returns a list; `[0]` takes its first item.

</details>

In [ ]:
def etl(customers):
    # Write code here: return a DataFrame
    return customers

etl(customers).show()

**Q6 · `etl(customers)`** Make a short name for every customer: the first name, a space, the first letter of the last name and a dot, like `Aziz K.` Show customer_id and short_name.

Expected output:

```
+-----------+----------+
|customer_id|short_name|
+-----------+----------+
|          1|   Aziz K.|
|          2|  Bella S.|
|          3| Carlos R.|
|          4|Dilnoza R.|
|          5|   Emma W.|
|          6|  Farid A.|
|          7|  Grace L.|
+-----------+----------+
```

<details><summary>Hint</summary>

`F.concat(...)` glues strings together; `F.substring(column, 1, 1)` takes the first letter; `F.lit` makes a fixed value.

</details>

In [ ]:
def etl(customers):
    # Write code here: return a DataFrame
    return customers

etl(customers).show()

**Q7 · `etl(customers)`** Mask the emails: keep the first letter, then `***`, then everything from the `@` on (like `a***@mail.uz`). Customers without an email drop out. Show name and masked_email.

Expected output:

```
+----------------+---------------+
|            name|   masked_email|
+----------------+---------------+
|    Aziz Karimov|   a***@mail.uz|
|   Bella Schmidt|    b***@web.de|
|     Carlos Ruiz| c***@correo.es|
|Dilnoza Rahimova|   d***@mail.uz|
|     Emma Wilson|e***@post.co.uk|
|       Grace Lee|    g***@web.de|
+----------------+---------------+
```

<details><summary>Hint</summary>

`F.substring(column, 1, 1)`, `F.substring_index(column, "@", -1)` (the part after the last `@`) and `F.concat`.

</details>

In [ ]:
def etl(customers):
    # Write code here: return a DataFrame
    return customers

etl(customers).show()

**Q8 · `etl(signups_raw)`** Keep only the digits of each phone number (`digits`), and its last 9 digits (`phone9`). Show signup_id, digits and phone9.

Expected output:

```
+---------+--------------+---------+
|signup_id|        digits|   phone9|
+---------+--------------+---------+
|        1|  998901234567|901234567|
|        2|00491512345678|512345678|
|        3|   34612345678|612345678|
|        4|     907654321|907654321|
|        5|   07700900123|700900123|
|        6|  998901234567|901234567|
+---------+--------------+---------+
```

<details><summary>Hint</summary>

`F.regexp_replace(column, "[^0-9]", "")` removes everything that is not a digit; `F.substring(column, -9, 9)` takes the last 9 characters.

</details>

In [ ]:
def etl(signups_raw):
    # Write code here: return a DataFrame
    return signups_raw

etl(signups_raw).show()

**Q9 · `etl(reviews)`** Clean the comments: remove the spaces at both ends, make them lower case, and remove every character that is not a letter, a digit or a space. Show review_id and clean_comment.

Expected output:

```
+---------+-----------------------------+
|review_id|clean_comment                |
+---------+-----------------------------+
|1        |great laptop very fast       |
|2        |nice story                   |
|3        |sound is ok delivery was slow|
|4        |broke after one week         |
|5        |fast and powerful            |
|6        |too long boring              |
|7        |useful recipes               |
|8        |good coffee slow start       |
+---------+-----------------------------+
```

<details><summary>Hint</summary>

`F.trim`, `F.lower`, then `F.regexp_replace(column, "[^a-z0-9 ]", "")`.

</details>

In [ ]:
def etl(reviews):
    # Write code here: return a DataFrame
    return reviews

etl(reviews).show(truncate=False)

**Q10 · `etl(customers)`** How many days had each customer been registered on 2024-04-01? Show name, signup_date and days_registered.

Expected output:

```
+----------------+-----------+---------------+
|            name|signup_date|days_registered|
+----------------+-----------+---------------+
|    Aziz Karimov| 2024-01-05|             87|
|   Bella Schmidt| 2024-01-20|             72|
|     Carlos Ruiz| 2024-02-02|             59|
|Dilnoza Rahimova| 2024-02-14|             47|
|     Emma Wilson| 2024-03-01|             31|
|    Farid Aliyev| 2024-03-10|             22|
|       Grace Lee| 2024-03-15|             17|
+----------------+-----------+---------------+
```

<details><summary>Hint</summary>

`F.datediff(end, start)`. Make the end date with `F.to_date(F.lit("2024-04-01"))`.

</details>

In [ ]:
def etl(customers):
    # Write code here: return a DataFrame
    return customers

etl(customers).show()

**Q11 · `etl(orders)`** For the orders placed in January and February, show order_id, order_date, the last day of that month (`month_end`) and the days left until then (`days_left`).

Expected output:

```
+--------+----------+----------+---------+
|order_id|order_date| month_end|days_left|
+--------+----------+----------+---------+
|    1001|2024-01-10|2024-01-31|       21|
|    1002|2024-01-10|2024-01-31|       21|
|    1003|2024-01-25|2024-01-31|        6|
|    1004|2024-02-03|2024-02-29|       26|
|    1005|2024-02-05|2024-02-29|       24|
|    1006|2024-02-15|2024-02-29|       14|
|    1007|2024-02-20|2024-02-29|        9|
+--------+----------+----------+---------+
```

<details><summary>Hint</summary>

`F.last_day(date)` and `F.datediff(end, start)`.

</details>

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q12 · `etl(signups_raw)`** The sign-up dates come in three formats (`2024-01-05`, `20/01/2024`, `2024/02/02`). Turn them into one date column `signup_day`. Show signup_id, signup_date and signup_day.

Expected output:

```
+---------+-----------+----------+
|signup_id|signup_date|signup_day|
+---------+-----------+----------+
|        1| 2024-01-05|2024-01-05|
|        2| 20/01/2024|2024-01-20|
|        3| 2024/02/02|2024-02-02|
|        4| 14/02/2024|2024-02-14|
|        5| 2024-03-01|2024-03-01|
|        6| 2024-03-02|2024-03-02|
+---------+-----------+----------+
```

<details><summary>Hint</summary>

`F.try_to_date(column, pattern)` returns null when the text doesn't match (plain `to_date` raises an error); `F.coalesce` takes the first one that worked.

</details>

In [ ]:
def etl(signups_raw):
    # Write code here: return a DataFrame
    return signups_raw

etl(signups_raw).show()

**Q13 · `etl(signups_raw)`** Turn amount (text) into a number `amount_value`; text that is not a number becomes null. Show signup_id, amount and amount_value.

Expected output:

```
+---------+------+------------+
|signup_id|amount|amount_value|
+---------+------+------------+
|        1| 12.50|        12.5|
|        2|   abc|        NULL|
|        3|     7|         7.0|
|        4|      |        NULL|
|        5|  3.75|        3.75|
|        6|  1.00|         1.0|
+---------+------+------------+
```

<details><summary>Hint</summary>

`Column.try_cast("double")` gives null instead of an error.

</details>

In [ ]:
def etl(signups_raw):
    # Write code here: return a DataFrame
    return signups_raw

etl(signups_raw).show()

**Q14 · `etl(customers, customers_new)`** List every customer who is in `customers` or in `customers_new`, once: customer_id and name, sorted by customer_id.

Expected output:

```
+-----------+----------------+
|customer_id|            name|
+-----------+----------------+
|          1|    Aziz Karimov|
|          2|   Bella Schmidt|
|          3|     Carlos Ruiz|
|          4|Dilnoza Rahimova|
|          5|     Emma Wilson|
|          6|    Farid Aliyev|
|          7|       Grace Lee|
|          8|       Hana Sato|
+-----------+----------------+
```

<details><summary>Hint</summary>

`unionByName` stacks two DataFrames with the same columns (matched by name); `distinct()` then removes the rows that are in both.

</details>

In [ ]:
def etl(customers, customers_new):
    # Write code here: return a DataFrame
    return customers

etl(customers, customers_new).show()

## Answers
Try each problem yourself first. Click a problem to open its answer.

<details>
<summary><b>Q1</b> &nbsp;Give every product a price_level: cheap if the price is under 50, mid from 50 to 200, premium above 200. Show ...</summary>

```python
def etl(products):
    return (products.withColumn("price_level",
                                F.when(F.col("price") < 50, "cheap")
                                 .when(F.col("price") <= 200, "mid")
                                 .otherwise("premium"))
                    .select("product_name", "price", "price_level"))

etl(products).show()
```

</details>

<details>
<summary><b>Q2</b> &nbsp;Show the orders that are shipped or cancelled and have a quantity of 2 or more: order_id, status and quantity.</summary>

```python
def etl(orders):
    return (orders.filter(F.col("status").isin("shipped", "cancelled") & (F.col("quantity") >= 2))
                  .select("order_id", "status", "quantity"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q3</b> &nbsp;Show customer_id, name and city of every customer, with Unknown where the city is missing. Call the name ...</summary>

```python
def etl(customers):
    return (customers.withColumnRenamed("name", "customer_name")
                     .withColumn("city", F.coalesce(F.col("city"), F.lit("Unknown")))
                     .select("customer_id", "customer_name", "city"))

etl(customers).show()
```

Also right: `customers.na.fill({"city": "Unknown"})`.

</details>

<details>
<summary><b>Q4</b> &nbsp;Which cities do the customers live in? One row per city, no empty values, from A to Z.</summary>

```python
def etl(customers):
    return customers.select("city").dropna().distinct().orderBy("city")

etl(customers).show()
```

</details>

<details>
<summary><b>Q5</b> &nbsp;Split name into first_name and last_name. Show customer_id, first_name and last_name.</summary>

```python
def etl(customers):
    parts = F.split(F.col("name"), " ")
    return customers.select("customer_id", parts[0].alias("first_name"), parts[1].alias("last_name"))

etl(customers).show()
```

</details>

<details>
<summary><b>Q6</b> &nbsp;Make a short name for every customer: the first name, a space, the first letter of the last name and a dot, ...</summary>

```python
def etl(customers):
    parts = F.split(F.col("name"), " ")
    short_name = F.concat(parts[0], F.lit(" "), F.substring(parts[1], 1, 1), F.lit("."))
    return customers.select("customer_id", short_name.alias("short_name"))

etl(customers).show()
```

</details>

<details>
<summary><b>Q7</b> &nbsp;Mask the emails: keep the first letter, then *, then everything from the @ on (like a*@mail.uz). Customers ...</summary>

```python
def etl(customers):
    return (customers.filter(F.col("email").isNotNull())
                     .select("name",
                             F.concat(F.substring("email", 1, 1), F.lit("***@"), F.substring_index("email", "@", -1)).alias("masked_email")))

etl(customers).show()
```

</details>

<details>
<summary><b>Q8</b> &nbsp;Keep only the digits of each phone number (digits), and its last 9 digits (phone9). Show signup_id, digits ...</summary>

```python
def etl(signups_raw):
    return (signups_raw.withColumn("digits", F.regexp_replace("phone", "[^0-9]", ""))
                       .select("signup_id", "digits", F.substring("digits", -9, 9).alias("phone9")))

etl(signups_raw).show()
```

</details>

<details>
<summary><b>Q9</b> &nbsp;Clean the comments: remove the spaces at both ends, make them lower case, and remove every character that is ...</summary>

```python
def etl(reviews):
    return (reviews.select("review_id",
                            F.regexp_replace(F.lower(F.trim("comment")), "[^a-z0-9 ]", "").alias("clean_comment")))

etl(reviews).show(truncate=False)
```

</details>

<details>
<summary><b>Q10</b> &nbsp;How many days had each customer been registered on 2024-04-01? Show name, signup_date and days_registered.</summary>

```python
def etl(customers):
    return (customers.withColumn("days_registered", F.datediff(F.to_date(F.lit("2024-04-01")), F.col("signup_date")))
                     .select("name", "signup_date", "days_registered"))

etl(customers).show()
```

</details>

<details>
<summary><b>Q11</b> &nbsp;For the orders placed in January and February, show order_id, order_date, the last day of that month ...</summary>

```python
def etl(orders):
    return (orders.filter(F.month("order_date") <= 2)
                  .select("order_id", "order_date",
                          F.last_day("order_date").alias("month_end"),
                          F.datediff(F.last_day("order_date"), "order_date").alias("days_left"))
                  .orderBy("order_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q12</b> &nbsp;The sign-up dates come in three formats (2024-01-05, 20/01/2024, 2024/02/02). Turn them into one date column ...</summary>

```python
def etl(signups_raw):
    return (signups_raw.select("signup_id", "signup_date",
                                F.coalesce(F.try_to_date("signup_date", "yyyy-MM-dd"),
                                           F.try_to_date("signup_date", "dd/MM/yyyy"),
                                           F.try_to_date("signup_date", "yyyy/MM/dd")).alias("signup_day")))

etl(signups_raw).show()
```

</details>

<details>
<summary><b>Q13</b> &nbsp;Turn amount (text) into a number amount_value; text that is not a number becomes null. Show signup_id, amount ...</summary>

```python
def etl(signups_raw):
    return signups_raw.select("signup_id", "amount", F.col("amount").try_cast("double").alias("amount_value"))

etl(signups_raw).show()
```

</details>

<details>
<summary><b>Q14</b> &nbsp;List every customer who is in customers or in customers_new, once: customer_id and name, sorted by ...</summary>

```python
def etl(customers, customers_new):
    return (customers.select("customer_id", "name")
                     .unionByName(customers_new.select("customer_id", "name"))
                     .distinct()
                     .orderBy("customer_id"))

etl(customers, customers_new).show()
```

`union` matches columns by position, `unionByName` by name: prefer `unionByName`, it can't silently mix up columns.

</details>